In [1]:
import os
import findspark
from pyspark.sql import SparkSession

os.environ['HADOOP_HOME'] = 'C:\\hadoop'
os.environ['hadoop.home.dir'] = 'C:\\hadoop'
os.environ['PATH'] = os.environ.get('PATH', '') + ';' + 'C:\\hadoop\\bin'

findspark.init()
spark = SparkSession.builder \
    .appName("CICIDS_XSS_Project_Standalone") \
    .master("spark://192.168.1.38:7077") \
    .config("spark.executor.memory", "3g") \
    .config("spark.driver.memory", "3g") \
    .config("spark.network.timeout", "800s") \
    .config("spark.executor.heartbeatInterval", "120s") \
    .getOrCreate()

print("Spark done")

Spark done


In [2]:
df = spark.read.csv("MachineLearningCSV/*.csv", header=True, inferSchema=True)
print("Initial statistics")
print("Number of rows:", df.count())
print("Number of columns:", len(df.columns))




Initial statistics
Number of rows: 2830743
Number of columns: 79


In [3]:
 df_clean = df.replace([float('inf'), float('-inf')],None).dropna()
df_clean = df_clean.toDF(*[c.strip() for c in df_clean.columns])
print("The data has been cleaned")
print("Number of rows remaining :", df_clean.count())

The data has been cleaned
Number of rows remaining : 2827876


In [4]:
from pyspark.sql.functions import col, when, trim

df_final = df_clean.withColumn(
    "label_idx", 
    when(trim(col("Label")).contains("XSS"), 1).otherwise(0)
)

distribution = (
    df_final
    .groupBy("Label", "label_idx")
    .count()
)

distribution.show(truncate=False)

total = df_final.count()
xss_count = df_final.filter(col("label_idx") == 1).count()

print("Number of rows :", total)
print("Number of XSS attacks :", xss_count)
print(f"Attack ratio : {(xss_count/total)*100:.4f} %")

+--------------------------+---------+-------+
|Label                     |label_idx|count  |
+--------------------------+---------+-------+
|BENIGN                    |0        |2271320|
|SSH-Patator               |0        |5897   |
|FTP-Patator               |0        |7935   |
|DoS GoldenEye             |0        |10293  |
|Heartbleed                |0        |11     |
|Infiltration              |0        |36     |
|Web Attack � XSS          |1        |652    |
|Bot                       |0        |1956   |
|Web Attack � Brute Force  |0        |1507   |
|Web Attack � Sql Injection|0        |21     |
|DoS Hulk                  |0        |230124 |
|DoS Slowhttptest          |0        |5499   |
|DoS slowloris             |0        |5796   |
|DDoS                      |0        |128025 |
|PortScan                  |0        |158804 |
+--------------------------+---------+-------+

Number of rows : 2827876
Number of XSS attacks : 652
Attack ratio : 0.0231 %


In [5]:
from pyspark.sql.functions import col, when
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

print("Building Pipeline + Class Weights")

total_rows = df_final.count()
xss_count = df_final.filter(col("label_idx") == 1).count()
other_count = total_rows - xss_count

weight_for_0 = total_rows / (2 * other_count)
weight_for_1 = total_rows / (2 * xss_count)

# Add weight column
df_weighted = df_final.withColumn(
    "class_weight",
    when(col("label_idx") == 1, weight_for_1).otherwise(weight_for_0)
)

print(f"Normal weight: {weight_for_0}")
print(f"XSS weight: {weight_for_1}")
print("-" * 50)


# Choose Features
selected_features = [
    'Destination Port',
    'Flow Duration',
    'Total Fwd Packets',
    'Fwd Packet Length Max',
    'Packet Length Mean'
]

#  Data preparation
assembler = VectorAssembler(
    inputCols=selected_features,
    outputCol="raw_features"
)

scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features"
)

pipeline = Pipeline(stages=[assembler, scaler])

# application
pipeline_model = pipeline.fit(df_weighted)
final_data = pipeline_model.transform(df_weighted)

print("Data ready")

# View Sample
final_data.select("label_idx", "class_weight", "features").show(5, truncate=False)


Building Pipeline + Class Weights
Normal weight: 0.5001153074535304
XSS weight: 2168.616564417178
--------------------------------------------------
Data ready
+---------+------------------+--------------------------------------------------------------------------------------------------------+
|label_idx|class_weight      |features                                                                                                |
+---------+------------------+--------------------------------------------------------------------------------------------------------+
|0        |0.5001153074535304|[2.691646028163989,1.1880892393100758E-7,0.0026664793790331493,0.008362156546311847,0.01963360276324381]|
|0        |0.5001153074535304|[2.691646028163989,2.9702230982751896E-8,0.0026664793790331493,0.008362156546311847,0.01963360276324381]|
|0        |0.5001153074535304|[2.691646028163989,2.9702230982751896E-8,0.0026664793790331493,0.008362156546311847,0.01963360276324381]|
|0        |0.50011530745

In [6]:
train_data, test_data = final_data.randomSplit([0.8, 0.2], seed=42)

print("Number of training data:", train_data.count())
print("Number of test data:", test_data.count())

Number of training data: 2262680
Number of test data: 565196


In [7]:
from pyspark.ml.classification import LogisticRegression
import time

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label_idx",
    weightCol="class_weight"
)

start = time.time()
lr_model = lr.fit(train_data)
lr_time = time.time() - start

print("Logistic Regression training")
print("Training time:", lr_time, "sec")

Logistic Regression training
Training time: 158.47076177597046 sec


In [8]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label_idx",
    weightCol="class_weight",     
    numTrees=50,
    seed=42    
)

start = time.time()
rf_model = rf.fit(train_data)
rf_time = time.time() - start

print("Random Forest training")
print("Training time:", rf_time, "sec")

Random Forest training
Training time: 302.10746812820435 sec


In [9]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol="label_idx",
    metricName="areaUnderROC"
)

# Logistic Regression
lr_pred = lr_model.transform(test_data)
lr_auc = evaluator.evaluate(lr_pred)

# Random Forest
rf_pred = rf_model.transform(test_data)
rf_auc = evaluator.evaluate(rf_pred)

print("results Logistic Regression:")
print("AUC:", lr_auc)
print("time:", lr_time)

print("-" * 40)

print("results Random Forest:")
print("AUC:", rf_auc)
print("time:", rf_time)

results Logistic Regression:
AUC: 0.933240234219932
time: 158.47076177597046
----------------------------------------
results Random Forest:
AUC: 0.996621708666029
time: 302.10746812820435


In [10]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator_precision = MulticlassClassificationEvaluator(
    labelCol="label_idx",
    metricName="precisionByLabel"
)

evaluator_recall = MulticlassClassificationEvaluator(
    labelCol="label_idx",
    metricName="recallByLabel"
)

# Logistic Regression
print("Logistic Regression:")
print("Precision (XSS):", evaluator_precision.evaluate(lr_pred, {evaluator_precision.metricLabel: 1}))
print("Recall (XSS):", evaluator_recall.evaluate(lr_pred, {evaluator_recall.metricLabel: 1}))

print("-" * 40)

# Random Forest
print("Random Forest:")
print("Precision (XSS):", evaluator_precision.evaluate(rf_pred, {evaluator_precision.metricLabel: 1}))
print("Recall (XSS):", evaluator_recall.evaluate(rf_pred, {evaluator_recall.metricLabel: 1}))

Logistic Regression:
Precision (XSS): 0.000534666124251758
Recall (XSS): 0.9517241379310345
----------------------------------------
Random Forest:
Precision (XSS): 0.018077546440593444
Recall (XSS): 1.0
